# Chapter 11: How Much Computation?

## Capacity as accumulated non-commutativity, dependency weighting, and plane decomposition

A transformer with $L$ layers performs $L$ successive rotations of the hidden state.  But how much *computational work* does the full stack actually do?  If every layer applied the same rotation, the network would be doing very little — a single rotation repeated.  The real work happens when successive rotations **fail to commute**: they push the hidden state through genuinely different geometric transformations.

In the Geometric Algebra framework, each layer transition $l \to l+1$ is characterized by a bivector generator $B^{(l)}$.  The **commutator** $[B^{(l)}, B^{(l+1)}] = B^{(l)} B^{(l+1)} - B^{(l+1)} B^{(l)}$ measures how much two consecutive rotations disagree about their planes and angles.

We define the **accumulated non-commutativity**:

$$C_{\mathrm{acc}} = \sum_{i < j} \bigl\| [B^{(i)},\, B^{(j)}] \bigr\|_F$$

This is a scalar summarizing the total geometric work done by the network.  If all layers commuted (rotated in exactly the same planes), $C_{\mathrm{acc}} = 0$ regardless of how large the individual rotations are.

But not all layers matter equally for the prediction.  The **dependency density** $D_l$ (from gradient-based analysis) tells us how much each layer influences the output.  Weighting by dependency gives the **effective capacity**:

$$C_{\mathrm{eff}} = \sum_{i < j} \bigl\| [B^{(i)},\, B^{(j)}] \bigr\|_F \,\sqrt{D_i \, D_j}$$

Finally, the commutator bivectors themselves can be **decomposed into principal planes**, revealing which geometric directions carry the most non-commutativity.

**By the end of this chapter you will be able to:**
- Compute accumulated and effective capacity for any prompt
- Identify which layers contribute most to computational work
- Decompose capacity into principal planes of non-commutativity
- Compare capacity across different task types
- Verify the Jacobi identity as a consistency check on real transformer bivectors

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga
from layer_time_ga.capacity import ga_capacity_profile
from layer_time_ga.curvature import commutator_plane_decomposition
from layer_time_ga.algebra import commutator_bivector

print("Imports OK")
print(f"NumPy {np.__version__}")

In [ ]:
# Load the model (this may take a minute on first run)
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")

print(f"Model: {model.name}")
print(f"Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")
print(f"Device: {model.device}")

## Accumulated Non-Commutativity

The central quantity in capacity analysis is the **commutator** of bivector generators:

$$[B^{(i)}, B^{(j)}] = B^{(i)} B^{(j)} - B^{(j)} B^{(i)}$$

For skew-symmetric matrices (our bivector representation), the commutator is itself skew-symmetric — i.e., another bivector.  Its Frobenius norm measures the *degree of non-commutativity* between layers $i$ and $j$.

The **accumulated non-commutativity** sums over all pairs:

$$C_{\mathrm{acc}} = \sum_{i < j} \bigl\| [B^{(i)},\, B^{(j)}] \bigr\|_F$$

**Key insight:** If all layers rotated in the same plane (even by different angles), their bivectors would be proportional and $[B^{(i)}, B^{(j)}] = 0$ for all pairs.  Non-zero $C_{\mathrm{acc}}$ means the network is rotating the hidden state through *genuinely different subspaces* at different depths.

The **effective capacity** $C_{\mathrm{eff}}$ weights each pair by the geometric mean of the dependency densities $\sqrt{D_i D_j}$, so that non-commutativity in gradient-irrelevant layers is downweighted.

The **concentration** $c_{\mathrm{conc}}$ measures what fraction of $C_{\mathrm{acc}}$ comes from the final layers — a high value means the network concentrates its non-trivial computation near the output.

In [ ]:
# Compute the full GA capacity profile for a single prompt
prompt = "The capital of France is"
cap = ltg_ga.capacity(prompt, model=model)

print(f"Prompt: \"{prompt}\"")
print(f"Number of bivectors (layer transitions): {len(cap.bivectors)}")
print()
print(f"=== Capacity Summary ===")
print(f"  C_acc  (accumulated non-commutativity): {cap.C_acc:.4f}")
print(f"  C_eff  (dependency-weighted capacity):  {cap.C_eff:.4f}")
print(f"  c_conc (final-layer concentration):     {cap.cconc:.4f}")
print()
print(f"Interpretation:")
if cap.C_eff > 0:
    ratio = cap.C_eff / cap.C_acc
    print(f"  C_eff / C_acc = {ratio:.4f}")
    print(f"  {'High' if ratio > 0.3 else 'Low'} fraction of geometric work is functionally relevant")
print(f"  Concentration = {cap.cconc:.1%} of non-commutativity in final layers")

## Per-Layer Contributions

The total capacity $C_{\mathrm{acc}}$ can be decomposed by layer.  The **per-layer contribution** of layer $i$ is:

$$C_i = \sum_{j \neq i} \bigl\| [B^{(i)},\, B^{(j)}] \bigr\|_F$$

This tells us how much layer $i$ "disagrees" with all other layers — i.e., how geometrically distinctive its rotation is.  Layers with high $C_i$ are doing something that no other layer does; layers with low $C_i$ are redundant (their rotations commute with most others).

This is a very different picture from just looking at rotation angles.  A layer can have a large rotation angle but low capacity contribution if it rotates in the same plane as its neighbors.

In [ ]:
# Bar chart of per-layer contributions to C_acc
fig, ax = plt.subplots(figsize=(10, 4))

n_layers = len(cap.layer_contributions)
layers = np.arange(n_layers)
colors = plt.cm.viridis(cap.layer_contributions / (cap.layer_contributions.max() + 1e-12))

ax.bar(layers, cap.layer_contributions, color=colors, edgecolor='none', width=0.8)

# Highlight the top-3 contributing layers
top3 = np.argsort(cap.layer_contributions)[-3:][::-1]
for rank, idx in enumerate(top3):
    ax.bar(idx, cap.layer_contributions[idx], color='#EE6677',
           edgecolor='black', linewidth=1.2, width=0.8)
    ax.annotate(f'#{rank+1}', xy=(idx, cap.layer_contributions[idx]),
                ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xlabel('Layer transition index')
ax.set_ylabel('Capacity contribution $C_i$')
ax.set_title(f'Per-Layer Capacity Contributions  ($C_{{\\mathrm{{acc}}}}$ = {cap.C_acc:.2f})')
ax.set_xlim(-0.5, n_layers - 0.5)
fig.tight_layout()

save_path = 'ch11_layer_contributions.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
print(f"\nTop-3 contributing layers: {top3.tolist()}")
for idx in top3:
    print(f"  Layer {idx}: C_i = {cap.layer_contributions[idx]:.4f} "
          f"({cap.layer_contributions[idx]/cap.layer_contributions.sum()*100:.1f}% of total)")
plt.close(fig)

## Plane Decomposition of Capacity

The commutator $[B^{(i)}, B^{(j)}]$ is itself a bivector — it lives in a specific plane (or superposition of planes).  By aggregating all pairwise commutator bivectors across the network, we can extract the **principal planes of non-commutativity**.

This is a powerful summary: it tells us not just *how much* the network's layers disagree, but *in which geometric directions* they disagree most.

The function `commutator_plane_decomposition` performs this aggregation:
1. Compute $[B^{(i)}, B^{(j)}]$ for all pairs $i < j$
2. Sum the absolute values of the commutator matrices (to avoid cancellation)
3. Re-skew-symmetrize and extract principal planes via eigendecomposition

Each principal plane comes with a **weight** (how much non-commutativity it carries) and **plane vectors** (the two directions spanning the plane).

In [ ]:
# Plane decomposition of the aggregate commutator structure
plane_info = commutator_plane_decomposition(cap.bivectors, n_planes=5)

print(f"Total commutator norm: {plane_info['total_norm']:.4f}")
print(f"Aggregate bivector norm: {plane_info['aggregate_bivector'].norm:.4f}")
print()

# Bar chart of top-5 plane weights
planes = plane_info['principal_planes']
n_show = min(5, len(planes))

fig, ax = plt.subplots(figsize=(8, 4))
plane_labels = [f"Plane {i+1}" for i in range(n_show)]
plane_weights = [planes[i]['weight'] for i in range(n_show)]
total_weight = sum(plane_weights)

bars = ax.bar(plane_labels, plane_weights, color='#4477AA', edgecolor='white', width=0.6)

# Annotate with percentage
for bar, w in zip(bars, plane_weights):
    pct = w / total_weight * 100 if total_weight > 0 else 0
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Plane weight (eigenvalue magnitude)')
ax.set_title('Principal Planes of Non-Commutativity')
fig.tight_layout()

save_path = 'ch11_capacity_planes.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")

print(f"\nTop-{n_show} principal planes of non-commutativity:")
for i in range(n_show):
    p = planes[i]
    print(f"  Plane {i+1}: weight = {p['weight']:.4f}, "
          f"angle = {p['angle']:.4f} rad")
plt.close(fig)

## Capacity Across Task Types

Different tasks require different amounts of computational work.  A simple factual recall ("The capital of France is") might need relatively little non-commutativity — the answer is stored in a localized region of the network.  Arithmetic or multi-step reasoning should require more, because the hidden state must be rotated through multiple distinct subspaces to compose the answer.

We can compare tasks by running the full analysis on each prompt and examining their rotor angle profiles.  The **mean rotation angle** $\bar{\theta} = \frac{1}{L}\sum_l \theta_l$ serves as a quick proxy for geometric activity, while the shape of the profile reveals where in the network the work happens.

In [ ]:
# Compare capacity across four task types
task_prompts = {
    "Factual":    "The capital of France is",
    "Arithmetic": "What is 347 plus 258? The answer is",
    "Reasoning":  "If all roses are flowers and some flowers fade quickly, then",
    "Creative":   "Write a short poem about the ocean at midnight:",
}

# Run analysis on each prompt (without dependency for speed)
task_results = {}
for label, prompt in task_prompts.items():
    print(f"Analysing: {label} ...")
    task_results[label] = ltg_ga.analyse(prompt, model=model, compute_dependency=False)

print("\nAll analyses complete.")

# Overlaid rotor angle profiles
fig, ax = plt.subplots(figsize=(10, 5))
colors_task = {'Factual': '#4477AA', 'Arithmetic': '#EE6677',
               'Reasoning': '#228833', 'Creative': '#AA3377'}

for label, r in task_results.items():
    ax.plot(r.angles, color=colors_task[label], linewidth=1.8,
            label=f"{label} ($\\bar{{\\theta}}$ = {r.angles.mean():.3f})", alpha=0.85)

ax.set_xlabel('Layer transition')
ax.set_ylabel('Rotation angle (rad)')
ax.set_title('Rotor Angle Profiles Across Task Types')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()

save_path = 'ch11_task_comparison.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")

# Print summary
print(f"\n{'Task':<14} {'Mean angle':>10} {'Max angle':>10} {'Peak layer':>11}")
print("-" * 48)
for label, r in task_results.items():
    print(f"{label:<14} {r.angles.mean():>10.4f} {r.angles.max():>10.4f} {r.angles.argmax():>11d}")
plt.close(fig)

## The Jacobi Identity as Quality Check

The **Jacobi identity** is a fundamental algebraic constraint on commutators.  For any three bivectors $A$, $B$, $C$:

$$[A,\,[B, C]] + [B,\,[C, A]] + [C,\,[A, B]] = 0$$

This holds *exactly* for matrix commutators, so it should hold (up to floating-point precision) for our bivector generators.  Verifying the Jacobi identity on real transformer bivectors serves as a **sanity check** that our extraction pipeline is numerically stable.

If the Jacobi residual $\|[A,[B,C]] + [B,[C,A]] + [C,[A,B]]\|_F$ is large relative to the individual commutator norms, something may be wrong with the numerical computation (e.g., rank truncation artifacts, degenerate layers).

This is also pedagogically interesting: the Jacobi identity is what makes the space of bivectors a **Lie algebra** under the commutator bracket.  The fact that it holds for real transformer data confirms that our GA framework is algebraically consistent.

In [ ]:
# Verify the Jacobi identity on real transformer bivectors
# Pick three bivectors from different parts of the network
bivectors = cap.bivectors
n_biv = len(bivectors)

# Choose early, middle, and late layers
idx_a, idx_b, idx_c = 0, n_biv // 2, n_biv - 1
A = bivectors[idx_a]
B = bivectors[idx_b]
C = bivectors[idx_c]

print(f"Testing Jacobi identity with bivectors from layers {idx_a}, {idx_b}, {idx_c}")
print(f"  ||A|| = {A.norm:.6f}")
print(f"  ||B|| = {B.norm:.6f}")
print(f"  ||C|| = {C.norm:.6f}")
print()

# Compute [A, [B, C]] + [B, [C, A]] + [C, [A, B]]
BC = commutator_bivector(B, C)
CA = commutator_bivector(C, A)
AB = commutator_bivector(A, B)

term1 = commutator_bivector(A, BC)
term2 = commutator_bivector(B, CA)
term3 = commutator_bivector(C, AB)

# The sum should be (approximately) zero
jacobi_sum = term1.matrix + term2.matrix + term3.matrix
jacobi_norm = float(np.linalg.norm(jacobi_sum, 'fro'))

# Scale for comparison
individual_norms = term1.norm + term2.norm + term3.norm
relative_residual = jacobi_norm / (individual_norms + 1e-15)

print(f"  ||[A,[B,C]]|| = {term1.norm:.6e}")
print(f"  ||[B,[C,A]]|| = {term2.norm:.6e}")
print(f"  ||[C,[A,B]]|| = {term3.norm:.6e}")
print()
print(f"  ||Jacobi sum|| = {jacobi_norm:.6e}")
print(f"  Relative residual = {relative_residual:.6e}")
print()

if relative_residual < 1e-8:
    print("  PASS: Jacobi identity holds to machine precision.")
elif relative_residual < 1e-4:
    print("  PASS: Jacobi identity holds within numerical tolerance.")
else:
    print("  WARNING: Jacobi residual is larger than expected — check numerics.")

## Exercises

1. **Capacity vs. angle.** For the four task prompts above, compute the full `ltg_ga.capacity()` profile (with dependency). Compare $C_{\mathrm{acc}}$ and $C_{\mathrm{eff}}$ across tasks. Does the ranking by $C_{\mathrm{eff}}$ match your intuition about task difficulty?

2. **Redundant layers.** Find layers where the per-layer contribution $C_i$ is less than 1% of the total. These layers commute with nearly all others. What does this suggest about network pruning?

3. **Plane stability.** Run the plane decomposition on multiple prompts. Do the same top planes appear across prompts, or are they prompt-specific? If the top planes are consistent, what does that tell us about the model's learned representations?

4. **Jacobi on random triples.** Repeat the Jacobi identity check with 10 randomly chosen triples of bivectors. Plot a histogram of the relative residuals. Is there a pattern (e.g., are early-layer bivectors more numerically stable)?

5. **Concentration and task type.** Compare $c_{\mathrm{conc}}$ (final-layer concentration) across the four task types. Do reasoning tasks show higher or lower concentration than factual recall? Why might this be the case?

6. **Scaling.** If you have access to models of different sizes (e.g., Qwen2.5-7B vs. Qwen2.5-32B), compare their $C_{\mathrm{acc}}$ on the same prompt. Does capacity scale with model size? How does $C_{\mathrm{eff}} / C_{\mathrm{acc}}$ change?